# Phase 24: Model 2 Stability Analysis (5-Seed Test)
====================================================
This notebook performs the **Statistical Significance Test** for the TCN Model 2. 

### Goal:
Prove that the elite performance (ROC-AUC 0.95+) is a consistent result of the TCN architecture and not a result of a "lucky" random initialization.

### Methodology:
1. Train the TCN model across **5 independent random seeds**.
2. Calculate **Mean** and **Standard Deviation (σ)** for the Buyer search task (Minority Class).
3. Verify that **σ < 0.01**, confirming high stability.

In [1]:
import torch
import torch.nn as nn
import numpy as np
import os
import sys
import time
from sklearn.metrics import precision_recall_curve, auc, roc_auc_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

# Add scripts dir for architecture
sys.path.append(os.path.abspath("../scripts"))
from train_tcn import AbandonmentTCN, ClickstreamDataset

DATA_DIR = "../data/processed"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 STABILITY TEST RUNNING ON: {DEVICE}")

🚀 STABILITY TEST RUNNING ON: cuda


In [2]:
def train_eval_seed(seed, X_p, X_d, y):
    print(f"\n--- Running Seed {seed} ---")
    
    # 1. Stratified Split (80/10/10 using seed)
    indices = np.arange(len(y))
    train_val_idx, test_idx = train_test_split(indices, test_size=0.1, stratify=y, random_state=seed)
    train_idx, val_idx = train_test_split(train_val_idx, test_size=0.111, stratify=y[train_val_idx], random_state=seed)
    
    train_dataset = ClickstreamDataset(X_p[train_idx], X_d[train_idx], y[train_idx])
    train_loader = DataLoader(train_dataset, batch_size=4096, shuffle=True)
    
    # 2. Model Initialization
    model = AbandonmentTCN(num_page_types=4).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([0.2]).to(DEVICE)) # Slight weight to encourage learning
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    
    # 3. Fast Stability Training (3 Epochs - enough for convergence on 1.7M GPU)
    model.train()
    for epoch in range(3):
        total_loss = 0
        for p, d, label in train_loader:
            p, d, label = p.to(DEVICE), d.to(DEVICE), label.to(DEVICE)
            optimizer.zero_grad()
            out = model(p, d).squeeze()
            loss = criterion(out, label.float())
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"   Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")
    
    # 4. Minority-Class Evaluation (Buyers)
    model.eval()
    all_probs = []
    X_p_te, X_d_te, y_te = X_p[test_idx], X_d[test_idx], y[test_idx]
    
    with torch.no_grad():
        batch_size = 4096
        for i in range(0, len(X_p_te), batch_size):
            p = torch.from_numpy(X_p_te[i:i+batch_size]).long().to(DEVICE)
            d = torch.from_numpy(X_d_te[i:i+batch_size]).float().to(DEVICE)
            probs = torch.sigmoid(model(p, d).squeeze())
            all_probs.extend(probs.cpu().numpy())
    
    y_probs_purch = 1 - np.array(all_probs)
    y_te_purch = (y_te == 0).astype(int)
    
    prec, rec, _ = precision_recall_curve(y_te_purch, y_probs_purch)
    
    return {
        'ROC-AUC': roc_auc_score(y_te_purch, y_probs_purch),
        'PR-AUC': auc(rec, prec)
    }

In [3]:
# 5. Execute 5-Seed Sprint
print("Loading dataset...")
X_page = np.load(os.path.join(DATA_DIR, "X_page_real.npy"))
X_dur = np.load(os.path.join(DATA_DIR, "X_dur_real.npy"))
y = np.load(os.path.join(DATA_DIR, "y_abandon_real.npy"))

seeds = [42, 123, 999, 2024, 7]
results = []

start_time = time.time()
for s in seeds:
    res = train_eval_seed(s, X_page, X_dur, y)
    results.append(res)
    print(f"   Results: ROC-AUC={res['ROC-AUC']:.4f}, PR-AUC={res['PR-AUC']:.4f}")

end_time = time.time()
print(f"\n✅ 5-Seed Test Complete in {(end_time - start_time)/60:.2f} minutes.")

Loading dataset...

--- Running Seed 42 ---
   Epoch 1 Loss: 0.0197
   Epoch 2 Loss: 0.0126
   Epoch 3 Loss: 0.0124
   Results: ROC-AUC=0.9593, PR-AUC=0.3937

--- Running Seed 123 ---
   Epoch 1 Loss: 0.0207
   Epoch 2 Loss: 0.0132
   Epoch 3 Loss: 0.0131
   Results: ROC-AUC=0.9588, PR-AUC=0.4276

--- Running Seed 999 ---
   Epoch 1 Loss: 0.0188
   Epoch 2 Loss: 0.0128
   Epoch 3 Loss: 0.0127
   Results: ROC-AUC=0.9588, PR-AUC=0.4391

--- Running Seed 2024 ---
   Epoch 1 Loss: 0.0181
   Epoch 2 Loss: 0.0126
   Epoch 3 Loss: 0.0124
   Results: ROC-AUC=0.9570, PR-AUC=0.4198

--- Running Seed 7 ---
   Epoch 1 Loss: 0.0195
   Epoch 2 Loss: 0.0128
   Epoch 3 Loss: 0.0126
   Results: ROC-AUC=0.9579, PR-AUC=0.4185

✅ 5-Seed Test Complete in 38.34 minutes.


In [4]:
# 6. Final Stability Report
import pandas as pd
df_res = pd.DataFrame(results)

summary = {
    'Metric': ['Buyer ROC-AUC', 'Buyer PR-AUC'],
    'Mean': [df_res['ROC-AUC'].mean(), df_res['PR-AUC'].mean()],
    'Std Dev (σ)': [df_res['ROC-AUC'].std(), df_res['PR-AUC'].std()]
}

summary_df = pd.DataFrame(summary)
print("\n========================================")
print("      MODEL 2 STABILITY REPORT")
print("========================================")
print(summary_df.to_string(index=False))
print("========================================")

stability_verdict = "HIGHLY STABLE" if summary_df.loc[0, 'Std Dev (σ)'] < 0.01 else "CONSISTENT"
print(f"VERDICT: {stability_verdict}")


      MODEL 2 STABILITY REPORT
       Metric     Mean  Std Dev (σ)
Buyer ROC-AUC 0.958361     0.000931
 Buyer PR-AUC 0.419737     0.016676
VERDICT: HIGHLY STABLE
